In [22]:
from dotenv import load_dotenv

load_dotenv()

True

In [23]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [24]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [25]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [26]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover cottage cheese and sweet pea. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Great idea—cottage cheese and peas pair nicely. Here are some simple recipe ideas you can make with just those two (plus common pantry items). I’ve noted what else you’d typically need.

1) Peas and Cottage Cheese Bowl (quick snack or light lunch)
- What you need: cottage cheese, peas (fresh or thawed frozen), olive oil or lemon juice, salt, pepper; optional herbs (dill, chives, parsley).
- How it works: mix cottage cheese with peas, drizzle with a little olive oil or a squeeze of lemon, season to taste. Top with chopped herbs if you have them.

2) Fresh Pea Frittata with Cottage Cheese (if you have eggs)
- What you need: eggs, cottage cheese, peas, optional onion or herbs, salt/pepper, a little oil/butter for the pan.
- How it works: whisk eggs, fold in cottage cheese and peas, season. Pour into a skillet and cook until set, finishing under a broiler or with a cover on low heat.

3) Creamy Pasta with Peas and Cottage Cheese
- What you need: pasta, peas, cottage cheese, a touch of milk

In [27]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover cottage cheese and sweet pea. What can I make?', additional_kwargs={}, response_metadata={}, id='cab4641b-f743-4d2f-b9de-25e11c5acb3c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 669, 'prompt_tokens': 201, 'total_tokens': 870, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EOgCyUm443XlVPLRnIsZodtmZ7pVT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a98a-93eb-74f2-b057-7a5c3bb3e101-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes using cottage cheese and peas'}, 'id': 'call_leOjUf4vrGLxf

In [ ]:
# question = HumanMessage(content="Can you suggest any Indian recipes that I can make with these ingredients?")

# response = agent.invoke(
#     {"messages": [question]},
#     config,  
# )

# pprint(response)

{'messages': [HumanMessage(content='I have some leftover cottage cheese and sweet pea. What can I make?', additional_kwargs={}, response_metadata={}, id='cab4641b-f743-4d2f-b9de-25e11c5acb3c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 669, 'prompt_tokens': 201, 'total_tokens': 870, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EOgCyUm443XlVPLRnIsZodtmZ7pVT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a98a-93eb-74f2-b057-7a5c3bb3e101-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes using cottage cheese and peas'}, 'id': 'call_leOjUf4vrGLxf

## Image Input

In [29]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [30]:
print(uploader.value)

()


In [31]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

IndexError: tuple index out of range

In [ ]:
config = {"configurable": {"thread_id": "1"}}
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Suggest me a recipe based on the ingredients in this image."},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config,  
)

print(response['messages'][-1].content)

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_krktcRm5H1z6mJNRrkcrK9fU, call_i4ENcci81oIY9sGbdobin1pA, call_kMFKmiZ07BivXRBvPKLsAcSJ", 'type': 'invalid_request_error', 'param': 'messages.[14].role', 'code': None}}